In [1]:
!pip install pymupdf 
!pip install contractions 
!pip install num2words
!pip install jiwer
#import num2words as n2w
import contractions as ctr
import pandas as pd
import pymupdf
import re
import string
import jiwer
from normalize_time_simple import normalize_times
#Set DF display option
pd.set_option('display.max_colwidth', None)


# 1. Extract Data from reference progress note (ASSUMED PDF) + Get Data from NurseGPT (assumed CSV file)


In [2]:
# Get refernce text:
#mapping_color_doc = pd.read_excel(r"C:\Users\Khuong Nguyen\Desktop\NurseGPT Eval\Automation Script\Simulate_Tier_Mapping_For_ProgressNote.xlsx")
#ref_notes = pymupdf.open(r"")
#hypo_notes = pd.read_csv(r"") #Assume NurseGPT's artifacts stored as csv/ xlsx

def reg_pattern_find(text, page_number):
    pattern_list = {
         "Reference Diagnoses": r"Diagnoses\s*:(.*?)(?=\n[A-Z][a-z]+\s*:|$)",
    }
    result = {}
    
    for field_name, pattern in pattern_list.items():
        match = re.search(pattern, text, re.DOTALL)
        if match:
            result[field_name] =  match.group(1) 
        else:
             result[field_name] = None
             print(f"Page {page_number+1} does not contain the pattern, please recheck this page content")
    return result 
             

# Extract content from PDFs -> Convert into DF

def note_extraction(note_doc, mapping_doc):
    rows = []
    for page_num, page in enumerate(note_doc, start = 1): #Loop through each progress note (1 note = 1 page)
        note_content = page.get_text() #Get ALL content fields (Diagnoses details, Date Created, Physicians ID,etc)
        note_type = mapping_color_doc.loc[mapping_color_doc['Page Number'] == page_num, 'Type'].values[0]
        
        field_results = reg_pattern_find(note_content, page_num)
        field_results["Page Number"] = page_num
        field_results['Note Type'] = note_type 
        rows.append(field_results)
        
        reference_note_df = pd.DataFrame(rows)
        
    return reference_note_df 



ref_notes = note_extraction(ref_notes, mapping_color_doc)
ref_notes

NameError: name 'ref_notes' is not defined

# 2. Data Preprocessing Steps

In [3]:
spoken_ref_note = pd.read_csv(r"C:\Users\Khuong Nguyen\Desktop\NurseGPT Eval\Automation Script\WER_Automation_Script\Data\nurse_spoken_script(Claude).csv", encoding='cp1252')
nurse_gpt_note = pd.read_excel(r"C:\Users\Khuong Nguyen\Desktop\NurseGPT Eval\Automation Script\WER_Automation_Script\Data\nurse_asr_transcription_for_WER_test.xlsx", index_col = False)

In [ ]:

def contraction(text):
    if isinstance(text, str):
        return ctr.fix(text)
    else:
        return text

def convert_number_words(match):
    number_str = match.group(0)
    number_int = int(number_str)
    return n2w.num2words(number_int)



def preprocessing_pipeline(df, column):
    df = df.copy()
    df[column] = df[column].str.lower()  # Decapitalization
    df[column] = df[column].str.replace('\n', ' ')  # Remove newline

    df[column] = df[column].apply(normalize_times)
    #df[column] = df[column].apply(lambda x: re.sub(r'\d+', convert_number_words, x) if isinstance(x, str) else x)


    df[column] = df[column].str.replace('-', ' ')
    df[column] = df[column].str.replace(':', ' ')
    df[column] = df[column].str.replace('/', 'over')
    df[column] = df[column].str.replace(r"'s\b", '', regex=True)
    # Contraction
    df[column] = df[column].apply(contraction)

    # Removal of punctuation
    punc_table = str.maketrans("", "", string.punctuation)
    df[column] = df[column].str.translate(punc_table)

    # Removal of white space
    df[column] = df[column].str.replace(r'\s+', ' ', regex=True)  # Extra white space
    df[column] = df[column].str.strip()  # Trailing white space

    return df

spoken_note_cleaned = preprocessing_pipeline(spoken_ref_note, 'Reference Diagnoses')
nurseGPT_transcript_cleaned = preprocessing_pipeline(nurse_gpt_note, 'Hypothesis Diagnoses')

In [5]:
spoken_note_cleaned

,Reference Diagnoses,Page Number,Type
0,at approximately 19 00 i responded to an emergency call bell near the third floor elevator upon arrival i found mrs jane doe that room number one twenty one eighty two years old lying on her back with a care aide supporting her head the care aide witnessed the fall and reported that the resident was reaching for the elevator button when she lost her balance and fell backward striking the back of her head on the tile floor mrs doe was alert but confused about what happened there was a three centimeter hematoma on the back of her head but no open wounds her glasgow coma scale was thirteen eye response four verbal response four and motor response five i checked her vitals blood pressure was one sixty over ninety heart rate eighty eight respiratory rate eighteen and oxygen saturation ninety six percent on room air i initiated neuro checks every fifteen minutes and gave her 06 00 fifty milligrams of tylenol per standing orders i paged the on call physician but did not get an immediate response since this was a witnessed fall with a head impact i called nine one one for hospital transfer i also notified mrs doe power of attorney that her husband george the resident was transferred to the hospital at seven thirty five pm along with her med administration record recent vitals and her advance directives i have let facility maintenance know to check the lighting in that elevator area i will follow up with the hospital for updates,1.0,Fall emergency
1,i found mrs jane doe on the floor by the third floor elevator after a witnessed fall she hit the back of her head on the floor she is awake but confused there is a three centimeter bump on the back of her head with no open cut her blood pressure is one sixty over ninety heart rate eighty eight breathing eighteen oxygen ninety six percent on room air i started neuro checks every fifteen minutes gave tylenol six fifty per standing order paged the doctor called nine one one and notified her husband george,2.0,Fall with a head injury
2,i did a routine morning check on mrs jane doe she is awake and knows where she is a little unsure of the time she looks comfortable her skin is warm dry and intact blood pressure one thirty two over seventy eight heart rate seventy two breathing sixteen temperature thirty six point eight oxygen ninety eight percent she had a normal bowel movement and is walking well with her walker no pain and no concerns,3.0,Routine morning check
3,recording vital signs for mrs jane doe blood pressure one twenty eight over seventy six heart rate seventy breathing sixteen temperature thirty six point seven oxygen ninety seven percent on room air she is comfortable with no complaints,4.0,Vital signs only
4,mrs jane doe has pain in both knees from her arthritis worse the last two days especially in the mornings she rates it six out of ten achy and stiff no swelling blood pressure one forty over eighty four heart rate seventy six temperature thirty six point seven oxygen ninety seven percent i gave acetaminophen six fifty per standing order used a warm compress and asked for a physiotherapy referral i notified her family and will check again in four hours,5.0,Knee pain
5,mrs jane doe is more short of breath than usual with little activity no chest pain or fever her oxygen is ninety one percent on room air down from ninety five yesterday breathing twenty two and a little labored with wheezing i gave ventolin two puffs with a spacer per order encouraged pursed lip breathing and paged the doctor i notified her family,6.0,Shortness of breath
6,a care aide found mrs jane doe unresponsive in her wheelchair by the first floor elevator pale and clammy she responded weakly to a sternal rub blood pressure ninety over sixty heart rate one twenty and irregular breathing twenty four oxygen ninety one percent i gave oxygen two liters by nasal cannula and her oxygen came up to ninety four i gave aspirin eighty one per order called the doctor and nine one one

In [6]:
print(spoken_ref_note['Reference Diagnoses'].iloc[0])

at approximately 19 00 i responded to an emergency call bell near the third floor elevator upon arrival i found mrs jane doe that room number one twenty one eighty two years old lying on her back with a care aide supporting her head the care aide witnessed the fall and reported that the resident was reaching for the elevator button when she lost her balance and fell backward striking the back of her head on the tile floor mrs doe was alert but confused about what happened there was a three centimeter hematoma on the back of her head but no open wounds her glasgow coma scale was thirteen eye response four verbal response four and motor response five i checked her vitals blood pressure was one sixty over ninety heart rate eighty eight respiratory rate eighteen and oxygen saturation ninety six percent on room air i initiated neuro checks every fifteen minutes and gave her 06 00 fifty milligrams of tylenol per standing orders i paged the on call physician but did not get an immediate respo

# Merging Data (Reference Spoken Notes + Hypothesis Notes)

In [6]:
import inspect
print(inspect.getsource(preprocessing_pipeline))

def preprocessing_pipeline(df, column):
    df[column] = df[column].str.lower()  # Decapitalization
    df[column] = df[column].str.replace('\n', ' ')  # Remove newline

    df[column] = df[column].apply(normalize_times)
    #df[column] = df[column].apply(lambda x: re.sub(r'\d+', convert_number_words, x) if isinstance(x, str) else x)


    df[column] = df[column].str.replace('-', ' ')
    df[column] = df[column].str.replace(':', ' ')
    df[column] = df[column].str.replace('/', 'over')
    df[column] = df[column].str.replace(r"'s\b", '', regex=True)
    # Contraction
    df[column] = df[column].apply(contraction)

    # Removal of punctuation
    punc_table = str.maketrans("", "", string.punctuation)
    df[column] = df[column].str.translate(punc_table)

    # Removal of white space
    df[column] = df[column].str.replace(r'\s+', ' ', regex=True)  # Extra white space
    df[column] = df[column].str.strip()  # Trailing white space

    return df



In [31]:
spoken_note_cleaned['Type'] = spoken_note_cleaned['Type'].str.strip()
nurseGPT_transcript_cleaned['Type'] = nurseGPT_transcript_cleaned['Type'].str.strip()

merged_df = pd.merge(spoken_note_cleaned, nurseGPT_transcript_cleaned, on=['Page Number', 'Type'], how='outer')

In [7]:
spoken_note_cleaned = preprocessing_pipeline(spoken_ref_note, 'Reference Diagnoses')
print(spoken_note_cleaned['Reference Diagnoses'].iloc[1])

i found mrs jane doe on the floor by the third floor elevator after a witnessed fall she hit the back of her head on the floor she is awake but confused there is a three centimeter bump on the back of her head with no open cut her blood pressure is one sixty over ninety heart rate eighty eight breathing eighteen oxygen ninety six percent on room air i started neuro checks every fifteen minutes gave tylenol six fifty per standing order paged the doctor called nine one one and notified her husband george


# WER Score Output

In [32]:
def wer_computation(df, ref_col, hyp_col):
    error_score_list =[]
    sub_error_list = []
    del_error_list = []
    insertions_error_list = []
    corr_count_list = []
    
    
    #Extract score
    for ref_text, hyp_text in zip(df[ref_col], df[hyp_col]):
        if pd.isna(ref_text) or pd.isna(hyp_text) is None:
            ref_text = ''
            hyp_text = ''   
        error_report = jiwer.process_words(ref_text, hyp_text)
        error_score = error_report.wer * 100
        sub_error = error_report.substitutions
        del_error = error_report.deletions
        insertions_error = error_report.insertions
        corr_count = error_report.hits

        # Append list
        error_score_list.append(error_score)
        sub_error_list.append(sub_error)
        del_error_list.append(del_error)
        insertions_error_list.append(insertions_error)
        corr_count_list.append(corr_count)

    new_cols = pd.DataFrame({
    'Error Score': error_score_list,
    'Substitution Error Count': sub_error_list,
    'Deletion Error Count': del_error_list,
    'Insertion Error Count': insertions_error_list,
    'Correct Transcribed Word Count': corr_count_list
    })

    df = pd.concat([df, new_cols], axis = 1)
    return df

df_wer_computed = wer_computation(merged_df, 'Reference Diagnoses', 'Hypothesis Diagnoses')
df_wer_computed

,Reference Diagnoses,Page Number,Type,Hypothesis Diagnoses,Error Score,Substitution Error Count,Deletion Error Count,Insertion Error Count,Correct Transcribed Word Count
0,at approximately nineteen zero i responded to an emergency call bell near the third floor elevator upon arrival i found mrs jane doe that room number one twenty one eighty two years old lying on her back with a care aide supporting her head the care aide witnessed the fall and reported that the resident was reaching for the elevator button when she lost her balance and fell backward striking the back of her head on the tile floor mrs doe was alert but confused about what happened there was a three centimeter hematoma on the back of her head but no open wounds her glasgow coma scale was thirteen eye response four verbal response four and motor response five i checked her vitals blood pressure was one sixty over ninety heart rate eighty eight respiratory rate eighteen and oxygen saturation ninety six percent on room air i initiated neuro checks every fifteen minutes and gave her six zero fifty milligrams of tylenol per standing orders i paged the on call physician but did not get an immediate response since this was a witnessed fall with a head impact i called nine one one for hospital transfer i also notified mrs doe power of attorney that her husband george the resident was transferred to the hospital at seven thirty five pm along with her med administration record recent vitals and her advance directives i have let facility maintenance know to check the lighting in that elevator area i will follow up with the hospital for updates,1.0,Fall emergency,at approximately seven of the clock pm i responded to an emergency call bell near the third floor elevator upon arrival i found mrs jane doe that room number one hundred and twenty one eighty two years old lying on her back with a care aide supporting her head the care aide witnessed the fall and reported that the resident was reaching for the elevator button when she lost her balance and fell backward striking the back of her head on the tile floor mrs doe was alert but confused about what happened there was a three centimeter he met oma on the back of her head but no open wounds her glasgow coma scale was thirteen eye response for verbal response for and motor response five i checked her vitals blood pressure was one hundred and sixty over ninety heart rate eighty eight respiratory rate eighteen and oxygen saturation ninety six on room air i initiated neuro checks every fifteen minutes and gave her six hundred and fifty milligrams of tylenol per standing orders i paged the on call physician but did not get an immediate response since this was a witnessed fall with a head impact i called nine hundred and eleven for hospital transfer i also notified mrs doe power of attorney that her husband george the resident was transferred to the hospital at nineteen thirty five along with her med administration record recent vitals and her advance directives i have let facility maintenance know to check the lighting in that elevator area i will follow up with the hospital for updates,8.593750,9,2,11,245
1,i found mrs jane doe on the floor by the third floor elevator after a witnessed fall she hit the back of her head on the floor she is awake but confused there is a three centimeter bump on the back of her head with no open cut her blood pressure is one sixty over ninety heart rate eighty eight breathing eighteen oxygen ninety six percent on room air i started neuro checks every fifteen minutes gave tylenol six fifty per standing order paged the doctor called nine one one and notified her husband george,2.0,Fall with a head injury,i found mrs jane doe on the floor by the third floor elevator after a witnessed fall she hit the back of her head on the floor she is awake but confused there is a three centimeter bump on the back of her head with no open cut her blood pressure is one hundred and sixty over ninety heart rate eight

In [33]:
import normalize_time_simple
print(normalize_time_simple.__file__)

from normalize_time_simple import normalize_times
print(normalize_times("at approximately seven o'clock pm i responded"))

c:\Users\Khuong Nguyen\Desktop\NurseGPT Eval\Automation Script\WER_Automation_Script\normalize_time_simple.py
at approximately 19:00 i responded
